In [ ]:
%reset -f
# 导入必要的库
import pyvisa
import os
import struct
import time
import function
from imp import reload
import function 
from importlib import reload
reload(function)
from RsSmw import *

# ============================
# 全局参数设置
# ============================

# 功能控制开关
ENABLE_SMW200A = 1           # 是否启用信号发生器
ENABLE_SWEEP = 1             # 是否执行扫频分析
ENABLE_ACPR = 0              # 是否执行邻道功率测量
ENABLE_IQ_ANALYSIS = 0       # 是否执行IQ分析
ENABLE_SAVE_IQ = 0           # 是否保存IQ数据到文件
ENABLE_PLOT_PSD = 0          # 是否绘制功率谱密度图
ENABLE_UPLOAD_WAVEFORM = 0   # 是否上传波形文件到信号发生器

# 设备IP地址
SMW200A_IP = "192.168.1.201"
CEYEAR_IP = "192.168.1.202"

# 频率和功率设置
CENTER_FREQ = "3.5GHz"                     # 中心频率
POWER_LEVEL = "-10"                        # 功率电平(dBm)
SAMPLING_RATE = "491.52MHz"                # 采样率
SAMPLING_RATE_HZ = 491.52e6                # 采样率(Hz)
MEASUREMENT_TIME = "0.001"                 # 测量时间(秒)
IQ_RECORD_LENGTH = "491520"                # IQ记录长度
SWEEP_RANGE = "500MHz"                     # 扫频分析中扫频带宽

# 频率和功率设置
CENTER_FREQ = "3.5GHz"                     # 中心频率
POWER_LEVEL = "-10"                        # 功率电平(dBm)
SAMPLING_RATE = "491.52MHz"                # 采样率
SAMPLING_RATE_HZ = 491.52e6                # 采样率(Hz)
MEASUREMENT_TIME = "0.001"                 # 测量时间(秒)
IQ_RECORD_LENGTH = "491520"                # IQ记录长度
SWEEP_RANGE = "500MHz"                     # 扫频分析中扫频带宽

RBW_OCCUPIED = "1MHz"                      # 占用带宽测量RBW
VBW_OCCUPIED = "3MHz"                      # 占用带宽测量VBW

# ACP测量参数
CARRIER_BW = "98.28MHz"                      # 载波带宽
ADJACENT_CH_OFFSET = "100MHz, 200MHz"        # 邻道偏移
ADJACENT_CH_BW = CARRIER_BW + ',' + CARRIER_BW  # 邻道带宽
RBW_ACLR = "100kHz"                        # ACLR测量RBW (满足3GPP要求)
VBW_ACLR = "300kHz"                        # ACLR测量VBW

# 文件路径和名称
# LOCAL_DIR = "D:/All_Projects/VsCode_projects/pa_auto_test/DPD_auto_test/data/3.5GHz/20250915"  # 本地目录
date = time.strftime("%Y%m%d", time.localtime()) + "/"
LOCAL_DIR = f"D:/All_Projects/VsCode_projects/pa_auto_test/DPD_auto_test/data/3.5GHz/{date.replace('/', '')}/"

DATA_FILE_NAME = "in1.txt"                    # 采集数据保存文件名
LOCAL_WV_FILE = LOCAL_DIR + 'arbFileExample.wv'  # 本地波形文件
INST_PRESET_WV_FILE = "USER/lg/NR-FR1-TM3.1a_100MHz_491.52Msps_30kHz_FDD_8.113dB.wv"  # 预设波形文件
# INST_WV_DIR = "USER/lg/20250915"             # 仪器上的目标目录
INST_WV_DIR = f"USER/lg/{date.replace('/', '')}"
INST_WV_FILE = INST_WV_DIR + "/arbFileExample.wv"  # 仪器上的波形文件




# 创建本地波形保存目录
os.makedirs(LOCAL_DIR, exist_ok=True)

# ============================
# Rohde & Schwarz SMW200A设置
# ============================

# 只有当启用信号发生器时才执行
if ENABLE_SMW200A:
    print("开始配置信号发生器...")
    
    # 创建VISA资源管理器
    rm_SMW200 = pyvisa.ResourceManager()

    # 连接到信号发生器
    usr_SMW200 = rm_SMW200.open_resource(f"TCPIP0::{SMW200A_IP}::inst0::INSTR")

    # 设置终止符和超时时间
    usr_SMW200.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)
    usr_SMW200.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR, ord('\n'))
    usr_SMW200.timeout = 30000

    # 关闭RF输出，配置基本设置
    usr_SMW200.write("OUTPut:STATe OFF")
    usr_SMW200.write(f"FREQ {CENTER_FREQ}")    # 设置频率
    usr_SMW200.write(f"POW {POWER_LEVEL}")     # 设置功率电平

    # 配置Marker和User连接器
    usr_SMW200.write("SOURce1:BB:ARBitrary:TRIGger:OUTPut1:MODE RESTart")
    usr_SMW200.write("OUTPut:USER1:DIRection OUTP")
    usr_SMW200.write("OUTPut:USER1:SIGNal MARKA1")
    usr_SMW200.write("OUTPut:USER4:DIRection OUTP")
    usr_SMW200.write("OUTPut:USER4:SIGNal MARKA1")

    # 尝试加载预设波形
    usr_SMW200.write(f'SOURce:BB:ARB:WAV:SELect "{INST_PRESET_WV_FILE}"')
    usr_SMW200.write("SOURce:BB:ARB:STATe ON")
    usr_SMW200.write("OUTPut:STATe ON")

    # 关闭连接
    usr_SMW200.close()
    rm_SMW200.close()
    
    print("信号发生器配置完成")
else:
    print("已跳过信号发生器配置")



# ============================
# 思仪频谱仪设置
# ============================

# 创建VISA资源管理器
rm_ceyear = pyvisa.ResourceManager()

# 连接到思仪频谱分析仪
usr_ceyear = rm_ceyear.open_resource(f"TCPIP0::{CEYEAR_IP}::5025::SOCKET")

# 设置终止符和超时时间
usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)
usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR, ord('\n'))
usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TMO_VALUE, 10000)

# ============================
# 思仪频谱仪扫频分析
# ============================
if ENABLE_SWEEP:
    # 配置扫频分析设置
    usr_ceyear.write(":INST SA")
    usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")
    usr_ceyear.write(f":FREQ:SPAN {SWEEP_RANGE}")    # 设置扫频范围
    
    # 设置RBW和VBW用于占用带宽测量
    usr_ceyear.write(f":BAND:RES {RBW_OCCUPIED}")    # 设置分辨带宽
    usr_ceyear.write(f":BAND:VID {VBW_OCCUPIED}")    # 设置视频带宽
    print(f"设置RBW: {RBW_OCCUPIED}, VBW: {VBW_OCCUPIED} (占用带宽测量)")
    for i in range(1, 7):
        usr_ceyear.write(f":TRAC{i}:DISP OFF")
    time.sleep(1)
    usr_ceyear.write(":TRAC1:DISP ON")
    usr_ceyear.write(":TRAC1:TYPE AVER")             # 显示方式
    usr_ceyear.write(":TRAC1:UPD ON")            # 开启曲线1更新
    time.sleep(5)
    usr_ceyear.write(":TRAC1:UPD OFF")
    usr_ceyear.write(":TRAC2:DISP ON")
    usr_ceyear.write(":TRAC2:UPD ON")            # 开启曲线1更新
    usr_ceyear.write(":TRAC2:TYPE AVER")             # 显示方式
    
    print("扫频分析完成")
else:
    print("\n已跳过扫频分析")

# ============================
# 邻道功率
# ============================
acp_results = None

if ENABLE_ACPR:
    print("\n开始执行邻道功率测量...")
    # 配置邻道功率测量
    usr_ceyear.write(":CONFigure:ACPower")
    time.sleep(1)
    usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")  
    # 设置无线电标准
    usr_ceyear.write(":RAD:STAN NR5GFR1B100M")  # 设置5G FR1带宽为100MHz
    
    # 可选自定义配置
    
    # usr_ceyear.write(f":ACP:CARR:LIST:BAND {CARRIER_BW}")
    # # #设置RBW和VBW用于ACLR测量
    # usr_ceyear.write(f":BAND:RES {RBW_ACLR}")        # 设置分辨带宽
    # usr_ceyear.write(f":BAND:VID {VBW_ACLR}")        # 设置视频带宽
    # print(f"设置RBW: {RBW_ACLR}, VBW: {VBW_ACLR} (ACLR测量)")
    # usr_ceyear.write(":ACP:OFFS:LIST:STAT 1,1,0,0,0,0,0,0")
    # usr_ceyear.write(f":ACP:OFFS:LIST {ADJACENT_CH_OFFSET}")
    # usr_ceyear.write(f":ACP:OFFS:LIST:BAND {ADJACENT_CH_BW}")

    usr_ceyear.write(":TRAC1:ACP:TYPE AVERage")

    # 查询ACP结果
    usr_ceyear.write(":FETC:ACP?")
    acp_results = usr_ceyear.read().strip().split(',')

    # 输出主要结果
    print(f"载波功率: {acp_results[0]} dBm")
    print(f"邻道: 低绝对: {acp_results[3]} dBm, 高绝对: {acp_results[5]} dBm")
    print(f"次邻道: 低绝对: {acp_results[7]} dBm, 高绝对: {acp_results[9]} dBm")
    print(f"邻道: 低相对: {acp_results[2]} dBm, 高相对: {acp_results[4]} dBm")
    print(f"次邻道: 低相对: {acp_results[6]} dBm, 高相对: {acp_results[8]} dBm")
else:
    print("\n已跳过邻道功率测量")



# ============================
# IQ分析
# ============================
iq_data = None
local_txt_path = os.path.join(LOCAL_DIR, DATA_FILE_NAME)

if ENABLE_IQ_ANALYSIS:
    print("\n开始执行IQ分析...")
    
    # 配置IQ分析
    usr_ceyear.write(":INST BASIC")
    usr_ceyear.write(f":FREQ:CENT {CENTER_FREQ}")
    usr_ceyear.write("ROSC:SOUR E10")
    usr_ceyear.write("TRIG:SOUR EXT1")
    usr_ceyear.write(f'TRAC:IQ:SRAT {SAMPLING_RATE}')
    usr_ceyear.write(f'SWE:TIME {MEASUREMENT_TIME}')
    usr_ceyear.write(f'TRAC:IQ:RLEN {IQ_RECORD_LENGTH}')
    usr_ceyear.write(':ADJ:LEV')
    time.sleep(5)

    # 执行测量
    usr_ceyear.write('INIT:IMM')
    usr_ceyear.clear()
    time.sleep(1)

    # 读取IQ数据
    usr_ceyear.timeout = 30000
    usr_ceyear.write(":FORM:DATA REAL,32")
    usr_ceyear.write(":TRAC:IQ:DATA?")

    # 禁用终止符检测并读取数据
    usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_FALSE)

    # 读取二进制块头
    header = usr_ceyear.read_bytes(2)
    digit_count = header[1] - 48
    length_str = usr_ceyear.read_bytes(digit_count).decode('ascii')
    data_length = int(length_str)

    # 分块读取数据
    raw_data = bytearray()
    chunk_size = 1024 * 1024
    remaining = data_length

    while remaining > 0:
        chunk = min(remaining, chunk_size)
        raw_data.extend(usr_ceyear.read_bytes(chunk))
        remaining -= chunk

    # 读取结束标志
    usr_ceyear.read_bytes(1)

    # 将二进制数据转换为浮点数数组
    float_count = len(raw_data) // 4
    iq_data = struct.unpack(f'{float_count}f', raw_data)

    # 恢复终止符检测
    usr_ceyear.set_visa_attribute(pyvisa.constants.VI_ATTR_TERMCHAR_EN, pyvisa.constants.VI_TRUE)

    # 保存IQ数据为TXT文件
    if ENABLE_SAVE_IQ:
        with open(local_txt_path, 'w') as f:
            for i in range(0, float_count, 2):
                if i+1 < float_count:
                    f.write(f"{iq_data[i]:.8f} {iq_data[i+1]:.8f}\n")
        print(f"IQ数据已保存到: {local_txt_path}")

    print(f"IQ数据采集完成，共{len(iq_data)//2}对")
else:
    print("\n已跳过IQ分析")

# 重新启用连续测量并关闭资源
usr_ceyear.write('INIT:CONT ON')
usr_ceyear.close()
rm_ceyear.close()

# 分析和可视化采集的IQ数据
if ENABLE_PLOT_PSD and (ENABLE_IQ_ANALYSIS or os.path.exists(local_txt_path)):
    print("\n绘制功率谱密度图...")
    data_ini_IQ, _, data_ini_cp = load_data(LOCAL_DIR, DATA_FILE_NAME)
    
    fig, axes = plt.subplots(1, 1, figsize=(20, 10))
    plot_PSD_func(data_ini_cp, data_ini_cp, SAMPLING_RATE_HZ, axes)
    plt.show()
    
    print("功率谱密度图绘制完成")
else:
    print("\n已跳过功率谱密度图绘制")

开始配置信号发生器...
信号发生器配置完成
设置RBW: 1MHz, VBW: 3MHz (占用带宽测量)
扫频分析完成

已跳过邻道功率测量

已跳过IQ分析

已跳过功率谱密度图绘制


In [ ]:
# 波形文件上传和激活

if ENABLE_UPLOAD_WAVEFORM:
    # 确保RsSmw库版本正确
    RsSmw.assert_minimum_version('5.0.44')

    # 创建连接
    smw = RsSmw(f'TCPIP0::{SMW200A_IP}::inst0::INSTR')
    rm = pyvisa.ResourceManager()
    usr_SMW200 = rm.open_resource(f"TCPIP0::{SMW200A_IP}::inst0::INSTR")
    usr_SMW200.timeout = 10000

    # 关闭RF输出
    usr_SMW200.write('OUTP:STAT OFF')

    # 创建目标目录
    usr_SMW200.write(f'MMEM:MDIR "{INST_WV_DIR}"')

    # 检查本地数据文件是否存在
    if os.path.exists(os.path.join(LOCAL_DIR, DATA_FILE_NAME)):
        # 加载IQ数据并创建波形文件
        data_ini_IQ, _, _ = load_data(LOCAL_DIR, DATA_FILE_NAME)

        i_data = data_ini_IQ[:,0]
        q_data = data_ini_IQ[:,1]

        # 创建波形文件
        smw.arb_files.create_waveform_file_from_samples(
            i_data, q_data, LOCAL_WV_FILE,
            clock_freq=SAMPLING_RATE_HZ, auto_scale=True,
            additional_tags=['MARKER LIST 1: 0:1;10:0'],
            comment='wv example'
        )

        # 上传波形文件到仪器
        with open(LOCAL_WV_FILE, 'rb') as file:
            file_content = file.read()

        data_len = len(file_content)
        len_str = str(data_len)
        header = f'#{len(len_str)}{len_str}'

        usr_SMW200.write_raw(f'MMEM:DATA "{INST_WV_FILE}",'.encode() + header.encode() + file_content)

        # 选择波形文件
        usr_SMW200.write(f'SOURce:BB:ARB:WAV:SELect "{INST_WV_FILE}"')

        # 启用ARB模式和RF输出
        usr_SMW200.write('SOURce:BB:ARB:STATe ON')
        usr_SMW200.write('OUTP:STAT ON')
        
        print("波形文件上传并激活完成")
    else:
        print(f"错误：找不到IQ数据文件 {os.path.join(LOCAL_DIR, DATA_FILE_NAME)}")
        print("波形文件上传失败")
    
    # 关闭连接
    usr_SMW200.close()
    rm.close()
else:
    print("\n已跳过波形文件上传")


波形文件上传并激活完成
